---
title: "GOES Solar Event Extractor"
subtitle: "Extract and format FLA / XRA events from GOES daily event listing files"
author: "SID Analysis Pipeline"
date: today
format:
  html:
    toc: true
    toc-depth: 3
    toc-title: "Contents"
    code-fold: show
    code-tools: true
    code-overflow: scroll
    theme: cosmo
    highlight-style: github
    self-contained: true
  pdf:
    toc: true
    toc-depth: 3
    colorlinks: true
    highlight-style: github
execute:
  echo: true
  warning: true
  error: false
jupyter: python3
---

## Overview

This notebook extracts **solar flare (FLA)** and/or **X-ray (XRA)** events
from GOES satellite daily event listing files and writes sorted, formatted
output files for downstream analysis (e.g. `goes_summ.py` or import into R).

It is a direct port of `1goesevents.py` to a literate, reproducible Quarto
document. Each section below corresponds to a logical stage of the pipeline.

---

## Directory Layout

The pipeline expects the following structure on disk. The `Data Received/`
directory is created automatically if it does not already exist.

```
<base_dir>/
└── <MMM_YY>/                       e.g.  JUN_11/
    ├── eventlistings/
    │   ├── yyyymmddevents.txt      e.g.  20110601events.txt
    │   └── ...
    └── Data Received/              ← output written here
        ├── JUN_11XRA.txt
        └── JUN_11FLA.txt
```

### Input file — fixed-width column layout

| Cols (0-based) | Field              |
|----------------|--------------------|
| 11 – 16        | Start time         |
| 17 – 23        | Peak time          |
| 27 – 33        | End time           |
| 43 – 47        | Event type (XRA / FLA) |
| 58 – 64        | Strength / class   |

### Output line format

```
MM DD     HHMMSS HHMMSS HHMMSS TYPE   CLASS
└─ date ─┘└start┘└peak ┘└end  ┘└type┘└class┘
```

Events are sorted chronologically and separated by a blank line between days.

---

## Imports

In [ ]:
import glob
import sys
from pathlib import Path

---

## Configuration {#sec-config}

Set the parameters for the run here.  If you are running interactively
set `INTERACTIVE = True` and the notebook will prompt you at each step.
For a scripted / batch run set `INTERACTIVE = False` and fill in the
values directly.

In [ ]:
# ── Run-mode toggle ──────────────────────────────────────────────────────────
INTERACTIVE: bool = False   # Set True to use interactive prompts instead

# ── Date period (used when INTERACTIVE = False) ──────────────────────────────
MONTH: str = "APR"          # Three-letter month, any case   e.g. "Jun", "JUN"
YEAR:  str = "26"           # Two-digit year                 e.g. "11"

# ── Base directory (used when INTERACTIVE = False) ───────────────────────────
# Leave as empty string "" to use the current working directory.
BASE_DIR: str = ""

# ── Event types to extract ───────────────────────────────────────────────────
# Any subset of {"FLA", "XRA"}.  Order does not matter.
QUAL_CODES: list[str] = ["XRA", "FLA"]

---

## Step 1 — Parse the Month / Year {#sec-parse-date}

`_parse_month_year` accepts any string whose first three characters are
letters (the month abbreviation) and whose last two characters are digits
(the two-digit year).  Separators between them are ignored, so `Jun11`,
`Jun 11`, and `Jun...11` are all valid.

In [ ]:
def _parse_month_year(raw: str) -> tuple[str, str]:
    """
    Parse a loose 'MMMnn' string into (MMM, YY).

    Parameters
    ----------
    raw : str
        User-supplied string, e.g. 'Jun11', 'Jun 11', 'jun_11', 'JUN...11'.

    Returns
    -------
    tuple[str, str]
        (mmm, yy) – both normalised: mmm is uppercase, yy is two digits.

    Raises
    ------
    ValueError
        If the month portion is not alphabetic or the year is not two digits.
    """
    raw = raw.strip()
    if len(raw) < 5:
        raise ValueError(f"Entry too short: '{raw}' — need at least 5 characters.")

    mmm = raw[:3]
    if not mmm.isalpha():
        raise ValueError(f"Month portion '{mmm}' must contain letters only.")

    yy = raw[-2:]
    if not yy.isdigit():
        raise ValueError(f"Year portion '{yy}' must be two digits.")

    return mmm.upper(), yy

### Smoke-test

In [ ]:
for sample in ["Jun11", "jun 11", "JUN...11", "jun_11"]:
    mmm, yy = _parse_month_year(sample)
    print(f"  '{sample}'  →  mmm={mmm!r}  yy={yy!r}")

---

## Step 2 — Resolve File Paths {#sec-paths}

`get_file_paths` combines the month/year with the base directory to build:

- **`event_glob`** — a wildcard pattern that matches every daily event file
- **`output_base`** — the path stem to which `XRA.txt` / `FLA.txt` are appended

In [ ]:
def get_file_paths(
    month_year_func=None,
    base_dir_func=None,
) -> tuple[str, Path]:
    """
    Resolve the event-listings glob pattern and output base path.

    In interactive mode the user is prompted; otherwise the values defined
    in the Configuration section are used directly.

    Parameters
    ----------
    month_year_func : callable, optional
        Override for the month/year prompt (used in unit tests).
    base_dir_func : callable, optional
        Override for the base-directory prompt (used in unit tests).

    Returns
    -------
    event_glob : str
        Glob pattern for all daily event files, e.g.
        '/data/JUN_11/eventlistings/*events.txt'
    output_base : Path
        Stem for output files, e.g. Path('/data/JUN_11/Data Received/JUN_11').
        The caller appends the qualifier code and '.txt'.

    Raises
    ------
    SystemExit
        If the eventlistings directory does not exist.
    """
    if INTERACTIVE:
        # ── Interactive branch ───────────────────────────────────────────────
        _prompt_my = month_year_func or (
            lambda: input(
                "\n Enter 3-character month and 2-digit year "
                "(any separator is fine):\n"
                " Examples:  Jun11   Jun 11   Jun...11\n > "
            )
        )
        _prompt_bd = base_dir_func or (
            lambda label: input(
                f"\n Path to the directory containing '{label}'"
                f" [press Enter for current directory]: "
            )
        )
        raw = _prompt_my()
        try:
            mmm, yy = _parse_month_year(raw)
        except ValueError as exc:
            sys.exit(f"\n Invalid date entry: {exc}\n")

        period_label = f"{mmm}_{yy}"
        raw_dir = _prompt_bd(period_label).strip()
        base_dir = Path(raw_dir).expanduser() if raw_dir else Path.cwd()

    else:
        # ── Scripted branch ──────────────────────────────────────────────────
        try:
            mmm, yy = _parse_month_year(MONTH + YEAR)
        except ValueError as exc:
            sys.exit(f"\n Configuration error: {exc}\n")

        period_label = f"{mmm}_{yy}"
        base_dir = Path(BASE_DIR).expanduser() if BASE_DIR else Path.cwd()

    period_dir   = base_dir / period_label
    listings_dir = period_dir / "eventlistings"
    output_dir   = period_dir / "Data Received"

    # Validate source directory.
    if not listings_dir.is_dir():
        sys.exit(
            f"\n Event listings directory not found:\n  {listings_dir}\n"
            "  Check that the path and period label are correct.\n"
        )

    # Create output directory if absent.
    output_dir.mkdir(parents=True, exist_ok=True)

    event_glob  = str(listings_dir / "*events.txt")
    output_base = output_dir / period_label

    return event_glob, output_base

---

## Step 3 — Select Event Qualifiers {#sec-qualifiers}

Valid event types are **`FLA`** (solar optical flares) and **`XRA`** (X-ray
events).  In scripted mode the `QUAL_CODES` list from @sec-config is used
directly; in interactive mode the user is prompted.

In [ ]:
def get_event_qualifiers(input_func=None) -> list[str]:
    """
    Return the list of event-type codes to extract.

    In interactive mode the user is prompted for a comma-separated list.
    In scripted mode the QUAL_CODES variable from the Configuration section
    is normalised and validated.

    Parameters
    ----------
    input_func : callable, optional
        Override for the qualifier prompt (used in unit tests).

    Returns
    -------
    list[str]
        Subset of ['FLA', 'XRA'] in canonical order.

    Raises
    ------
    SystemExit
        If no valid code is found.
    """
    VALID = {"FLA", "XRA"}

    if INTERACTIVE:
        _prompt = input_func or (
            lambda: input(
                "\n\n  Available event types: FLA  XRA\n"
                "  Enter one or both, comma-separated (e.g.  XRA,FLA): "
            )
        )
        raw    = _prompt().upper()
        tokens = {t.strip() for t in raw.split(",")}
    else:
        tokens = {c.strip().upper() for c in QUAL_CODES}

    codes = [c for c in ("FLA", "XRA") if c in tokens]

    if not codes:
        sys.exit(
            "\n No valid event qualifiers found. "
            "Please specify FLA, XRA, or both.\n"
        )

    return codes

### Smoke-test

In [ ]:
# Temporarily override INTERACTIVE to exercise the scripted path
_saved = INTERACTIVE
INTERACTIVE = False
print("Scripted codes:", get_event_qualifiers())
INTERACTIVE = _saved

---

## Step 4 — Extract Events from a Single File {#sec-extract}

`_extract_events` reads one daily file and returns a list of formatted
output lines for events matching the requested qualifier code.

In [ ]:
def _extract_events(filepath: str, code: str) -> list[str]:
    """
    Read one GOES daily event file and return formatted lines for *code* events.

    Parameters
    ----------
    filepath : str
        Full path to a daily event file, e.g. '.../20110601events.txt'.
    code : str
        Event qualifier to match, e.g. 'XRA' or 'FLA'.

    Returns
    -------
    list[str]
        Formatted event strings, one per matching line.  Empty list if none.

    Notes
    -----
    Output line format (0-based columns, space-separated):

    +---------+------------------+---------------------------------------------+
    | Cols    | Content          | Source                                      |
    +=========+==================+=============================================+
    |  0 –  4 | Month day        | Derived from filename chars 4-7             |
    |  5 – 10 | Padding          | Literal spaces                              |
    | 11 – 16 | Start time       | Input cols 11-16                            |
    | 17 – 23 | Peak time        | Input cols 17-23                            |
    | 24 – 30 | End time         | Input cols 27-33                            |
    | 31 – 35 | Event type       | Input cols 43-47                            |
    | 36 – 42 | Strength / class | Input cols 58-64                            |
    +---------+------------------+---------------------------------------------+
    """
    results: list[str] = []
    path = Path(filepath)
    stem = path.stem                        # e.g. '20110601events'

    # Month and day come from the filename, not the file body.
    try:
        month_day = f"{stem[4:6]} {stem[6:8]}"
    except IndexError:
        print(f"  Warning: unexpected filename format '{path.name}' — skipping.")
        return results

    try:
        with open(filepath, encoding="utf-8", errors="replace") as fh:
            for line in fh:
                # Lines shorter than 65 characters cannot contain all fields.
                if len(line) < 65:
                    continue
                if line[43:46] == code:
                    formatted = (
                        f"{month_day}     "
                        f"{line[11:17]}"
                        f"{line[17:24]}"
                        f"{line[27:34]}"
                        f"{line[43:48]}"
                        f"{line[58:65]}"
                    )
                    results.append(formatted.rstrip())
    except OSError as exc:
        print(f"  Warning: could not read '{filepath}': {exc}")

    return results

---

## Step 5 — Write Output File {#sec-write}

`_write_output` sorts all collected events chronologically and writes them to
disk, inserting a blank line between each change of day.

In [ ]:
def _write_output(output_path: Path, events: list[str], code: str) -> None:
    """
    Write a sorted list of event strings to *output_path*.

    Parameters
    ----------
    output_path : Path
        Destination file, e.g. Path('.../JUN_11XRA.txt').
    events : list[str]
        Formatted event strings as returned by _extract_events().
    code : str
        Event type label written to the file when no events are found.

    Notes
    -----
    Sort key is cols 3-13 of the output line (month day + start time),
    which gives strict chronological ordering across all input files.
    A blank line is inserted whenever the day field (cols 3-4) changes.
    """
    with open(output_path, "w", encoding="utf-8") as fh:
        if not events:
            fh.write(f"No valid {code} events found\n")
            return

        events.sort(key=lambda s: s[3:14])

        current_day: str | None = None
        for entry in events:
            day = entry[3:5]
            if current_day is not None and day != current_day:
                fh.write("\n")
            current_day = day
            fh.write(f"{entry}\n")

---

## Step 6 — Process All Files {#sec-process}

`process_files` is the main loop. It iterates over every qualifier code,
collects events from all matching daily files, and calls `_write_output`.

In [ ]:
def process_files(
    event_glob: str,
    output_base: Path,
    qual_codes: list[str],
) -> None:
    """
    Extract and write events for each requested qualifier code.

    Parameters
    ----------
    event_glob : str
        Glob pattern matching all daily event listing files.
    output_base : Path
        Output file stem; qualifier code and '.txt' are appended.
    qual_codes : list[str]
        Event types to extract, e.g. ['XRA', 'FLA'].
    """
    daily_files = sorted(glob.glob(event_glob))

    if not daily_files:
        print(f"\n Warning: no event files matched:\n  {event_glob}\n")
        return

    print(f"\n Found {len(daily_files)} daily event file(s).\n")

    for code in qual_codes:
        events: list[str] = []
        for filepath in daily_files:
            events.extend(_extract_events(filepath, code))

        output_path = Path(str(output_base) + code + ".txt")
        _write_output(output_path, events, code)
        print(f"  [{code}]  {len(events):4d} events  →  {output_path}")

---

## Step 7 — Run the Pipeline {#sec-run}

This cell ties everything together. It mirrors the `if __name__ == "__main__"`
block of the original script. Edit the @sec-config section above and then
re-run this cell (or render the whole document) to process a new period.

In [ ]:
def process_goes_events(output_base: Path | None = None) -> None:
    """
    Top-level entry point.

    Parameters
    ----------
    output_base : Path, optional
        If provided, overrides the output path derived from the period label.
        Useful when calling this notebook as a module.
    """
    event_glob, output_base_resolved = get_file_paths()
    qual_codes = get_event_qualifiers()

    if output_base is not None:
        output_base_resolved = output_base

    process_files(event_glob, output_base_resolved, qual_codes)


# ── Execute ──────────────────────────────────────────────────────────────────
process_goes_events()

---

## Command-Line Usage

When this notebook is *rendered* by Quarto the pipeline runs automatically
using the values in @sec-config.  The original script's CLI interface is
preserved below for users who prefer to run the extracted `.py` file directly.

```bash
# Interactive mode (prompts for month, year, directory, and event types)
python goes_events.py

# Scripted mode with an explicit output base path
python goes_events.py -o /path/to/output/JUN_11
```

Quarto can also render from the command line, which re-runs the full pipeline
and produces a self-contained HTML or PDF report:

```bash
# Render to HTML (default)
quarto render goes_events.qmd

# Render to PDF
quarto render goes_events.qmd --to pdf

# Pass a custom output base via Quarto parameters (requires params: block)
quarto render goes_events.qmd -P base_dir:/data -P month:JUN -P year:11
```

---

## Session Information

In [ ]:
import platform, sys as _sys
print(f"Python : {_sys.version}")
print(f"Platform: {platform.platform()}")